# Mechanisme : Ruben Snijders en Witse Panneels

mechanisme: 18 (R) + 23 (W) = 41 + 1 => **2** => Pantograaf

In [11]:
# imports and initialization

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

%matplotlib widget

In [12]:
# zetten van de kinematische parameters
# Alle eenheden zijn in meter of radialen

phi0 = 0 #rad
d1 = 1.6 #m
d2 = 1.1 #m
d3 = 0.8 #m

r1 = 1 #m
r2 = 1.5 #m
r3 = 1.05 #m
r4 = 3.3 #m
r5 = 2.4 #m
r6 = 1.3 #m
r7 = 5 #m

l41 = 1.35 #m
l51 = 1.1 #m
l71 = 2.2 #m

# l42, l52 en l72 bekomen we door rx - lx1
l42 = r4 - l41
l52 = r5 - l51
l72 = r7 - l71

#de beginwaarden voor de heoken voor fsolve

phi2_init = 4.2
phi3_init = 0.7
phi4_init = 5.6
phi5_init = 5.2
phi6_init = 5.2
phi7_init = 3.9

# Tijd + simulatie parameters
t_begin =  0     # start time of simulation
t_end   = 25     # end time of simulation
Ts      =  0.10  # time step of simulation
t = np.arange(t_begin, t_end + Ts, Ts)  # time vector

#initialization of driver
omega = 0.5      # driver frequency
A     = 1        # amplitude
# angular position, velocity and acceleration:
phi1   = 4.4 + A * np.sin(omega * t)
dphi1  = omega * A * np.cos(omega * t)
ddphi1 = -omega ** 2 * A * np.sin(omega * t)


In [13]:
# Function to rotate a vector z over an angle theta:
def rotate_vector(z, theta):
    rotation_matrix = np.array([[np.cos(theta), -np.sin(theta)],
                                [np.sin(theta), np.cos(theta)]])
    return np.dot(rotation_matrix, z)

In [14]:
# Define loop closure equations
def loop_closure_eqs(phi_init, phi1, d1, d2, d3, r1, r2, r3, r6, l41, l42, l51, l52, l71, phi0):
    phi2 = phi_init[0]
    phi3 = phi_init[1]
    phi4 = phi_init[2]
    phi5 = phi_init[3]
    phi6 = phi_init[4]
    phi7 = phi_init[5]

    # Loop closure equations
    F1 = r1 * np.cos(phi1) + l41 * np.cos(phi4) - r2 * np.cos(phi2) - d1 * np.cos(phi0)
    F2 = r1 * np.sin(phi1) + l41 * np.sin(phi4) - r2 * np.sin(phi2) - d1 * np.sin(phi0)
    F3 = r2 * np.cos(phi2) + l42 * np.cos(phi4) - r3 * np.cos(phi3) - d2 * np.cos(phi0) - d3 * np.sin(phi0) - l51 * np.cos(phi5)
    F4 = r2 * np.sin(phi2) + l42 * np.sin(phi4) - r3 * np.sin(phi3) - d2 * np.sin(phi0) + d3 * np.cos(phi0) - l51 * np.sin(phi5)
    F5 = r3 * np.cos(phi3) + l52 * np.cos(phi5) + l71 * np.cos(phi7) - r6 * np.cos(phi6)
    F6 = r3 * np.sin(phi3) + l52 * np.sin(phi5) + l71 * np.sin(phi7) - r6 * np.sin(phi6)

    return [F1, F2, F3, F4, F5, F6]

In [ ]:
# Ddefinieer de kinematische berekeningen van de pantograaf

def kinematics_pantograaf(d1, d2, d3, r1, r2, r3, r6, l41, l42, l51, l52, l71, phi0, phi1, dphi1, ddphi1, 
                          phi2_init, phi3_init, phi4_init, phi5_init, phi6_init, phi7_init, t):
    global phi2, phi3, phi4, phi5, phi6, phi7, dphi2, dphi3, dphi4, dphi5, dphi6, dphi7, ddphi2, ddphi3, ddphi4, ddphi5, ddphi6, ddphi7, cond
    phi2   = np.zeros_like(t)
    phi3   = np.zeros_like(t)
    phi4   = np.zeros_like(t)
    phi5   = np.zeros_like(t)
    phi6   = np.zeros_like(t)
    phi7   = np.zeros_like(t)
    dphi2   = np.zeros_like(t)
    dphi3   = np.zeros_like(t)
    dphi4   = np.zeros_like(t)
    dphi5   = np.zeros_like(t)
    dphi6   = np.zeros_like(t)
    dphi7   = np.zeros_like(t)
    ddphi2   = np.zeros_like(t)
    ddphi3   = np.zeros_like(t)
    ddphi4   = np.zeros_like(t)
    ddphi5   = np.zeros_like(t)
    ddphi6   = np.zeros_like(t)
    ddphi7   = np.zeros_like(t)

    cond   = np.zeros_like(t)

    optim_options = {"full_output":True}  # options for fsolve

    for k, time in enumerate(t):
        # Position Analysis
        x, _, ier, message  = fsolve(lambda x: loop_closure_eqs(x, phi1[k], d1, d2, d3, r1, r2, r3, r6, l41, l42, l51, l52, l71, phi0), 
                                     [phi2_init, phi3_init, phi4_init, phi5_init, phi6_init, phi7_init], **optim_options)

        if ier != 1:
            print("The fsolve exit flag was not 1, probably no convergence!")
            print(message)

        phi2[k] = x[0]
        phi3[k] = x[1]
        phi4[k] = x[2]
        phi5[k] = x[3]
        phi6[k] = x[5]
        phi7[k] = x[6]


        # Velocity Analysis
        A = np.array([[r2 * np.sin(phi2[k]), 0, -l41 * np.sin(phi4[k]), 0, 0, 0],
                      [-r2 * np.cos(phi2[k]),0, l41 * np.cos(phi4[k]), 0, 0, 0],
                      [-r2 * np.sin(phi2[k]), -r3 * np.sin(phi3[k]), -l42 * np.sin(phi4[k]), l51 * np.sin(phi5[k]), 0, 0],
                      [r2 * np.cos(phi2[k]), r3 * np.cos(phi3[k]), l42 * np.cos(phi4[k]), -l51 * np.sin(phi5[k]), 0, 0],
                      [0, -r3 * np.sin(phi3[k]), 0, -l52 * np.sin(phi5[k]), r6 * np.sin(phi6[k]), -l71 * np.sin(7[k])],
                      [0, r3 * np.cos(phi3[k]), 0, l52 * np.cos(phi5[k]), -r6 * np.cos(phi6[k]), l71 * np.cos(7[k])]])
        B = np.array([r1 * np.sin(phi1[k]) * dphi1[k],
                      -r1 * np.cos(phi1[k]) * dphi1[k]])

        x = np.linalg.solve(A, B)
        dphi2[k] = x[0]
        dphi3[k] = x[1]
        dphi4[k] = x[2]
        dphi5[k] = x[3]
        dphi6[k] = x[5]
        dphi7[k] = x[6]
        cond[k]  = np.linalg.cond(A)

        # Acceleration Analysis
        B = np.array([r1 * np.cos(phi1[k]) * dphi1[k]**2 + r1 * np.sin(phi1[k]) * ddphi1[k] - r2 * np.cos(phi2[k]) * dphi2[k]**2 + l41 * np.cos(phi4[k]) * dphi4[k]**2,
                      r1 * np.sin(phi1[k]) * dphi1[k]**2 - r1 * np.cos(phi1[k]) * ddphi1[k] - r2 * np.sin(phi2[k]) * dphi2[k]**2 + l41 * np.sin(phi4[k]) * dphi4[k]**2,
                      r2 * np.cos(phi2[k]) * dphi2[k]**2 + r3 * np.cos(phi3[k]) * dphi3[k]**2 + l42 * np.cos(phi4[k]) * dphi4[k]**2 - l51 * np.cos(phi5[k]) * dphi5[k]**2,
                      r2 * np.sin(phi2[k]) * dphi2[k]**2 + r3 * np.sin(phi3[k]) * dphi3[k]**2 + l42 * np.sin(phi4[k]) * dphi4[k]**2 - l51 * np.sin(phi5[k]) * dphi5[k]**2,
                      r3 * np.cos(phi3[k]) * dphi3[k]**2 + l52 * np.cos(phi5[k]) * dphi5[k]**2 - r6 * np.cos(phi6[k]) * dphi6[k]**2 + l71 * np.cos(phi7[k]) * dphi7[k]**2
                      r3 * np.sin(phi3[k]) * dphi3[k]**2 + l52 * np.sin(phi5[k]) * dphi5[k]**2 - r6 * np.sin(phi6[k]) * dphi6[k]**2] + l71 * np.sin(phi7[k]) * dphi7[k]**2])

        x = np.linalg.solve(A, B)
        ddphi2[k] = x[0]
        ddphi3[k] = x[1]
        ddphi4[k] = x[2]
        ddphi5[k] = x[3]
        ddphi6[k] = x[5]
        ddphi7[k] = x[6]

        # Next iteration step
        phi2_init = phi2[k] + (t[1] - t[0]) * dphi2[k]
        phi3_init = phi3[k] + (t[1] - t[0]) * dphi3[k]
        phi4_init = phi4[k] + (t[1] - t[0]) * dphi4[k]
        phi5_init = phi5[k] + (t[1] - t[0]) * dphi5[k]
        phi6_init = phi6[k] + (t[1] - t[0]) * dphi6[k]
        phi7_init = phi7[k] + (t[1] - t[0]) * dphi7[k]